# 02 — Görev 2: Özellik Seçimi & Normalizasyon

1. `StandardScaler` (yalnızca sürekli kolonlarda, train üzerinde fit)
2. Üç farklı özellik seçim yöntemi:
   - **Filter:** SelectKBest + ANOVA F-test (`f_classif`)
   - **Wrapper:** Recursive Feature Elimination (RFE) + LogisticRegression
   - **Embedded:** Random Forest `feature_importances_`
3. Üç yöntemin sonuçlarını karşılaştır → ortak / oybirliğiyle seçilenleri belirle
4. Karşılaştırma tablosunu CSV + MD olarak kaydet
5. Yöntemlerin kesişimini gösteren Venn diyagramı + yatay bar grafik


In [1]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd() / "src"))
from src import config as C
from src import utils as U

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

np.random.seed(C.RANDOM_STATE)

## 1. İşlenmiş veriyi yükle

In [2]:
X_train = pd.read_csv(C.PROC_DIR / "X_train.csv")
X_test = pd.read_csv(C.PROC_DIR / "X_test.csv")
y_train = pd.read_csv(C.PROC_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(C.PROC_DIR / "y_test.csv").squeeze("columns")
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
X_train.head()

Train: (227, 20)  |  Test: (61, 20)


,age,trestbps,chol,thalach,oldpeak,sex_1,cp_2,cp_3,cp_4,fbs_1,restecg_1,restecg_2,exang_1,slope_2,slope_3,ca_1.0,ca_2.0,ca_3.0,thal_6.0,thal_7.0
0,48,124,274,166,0.5,1,0,0,1,0,0,1,0,1,0,0,0,0,0,1
1,55,130,262,155,0.0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,54,132,288,159,0.0,0,1,0,0,1,0,1,1,0,0,1,0,0,0,0
3,54,108,309,156,0.0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,57,140,241,123,0.2,0,0,0,1,0,0,0,1,1,0,0,0,0,0,1


## 2. StandardScaler — sadece sürekli kolonlar

One-hot kolonlarını (0/1) scale etmek anlamsız, bilgi kaybettirmiyor ama
ağaç bazlı olmayan modellerin yorumunu zorlaştırıyor. Bu yüzden sadece sürekli olanları normalize ediyoruz.


In [3]:
scaler = StandardScaler()

# Sadece sürekli kolonları scale et
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[C.CONTINUOUS] = scaler.fit_transform(X_train[C.CONTINUOUS])
X_test_scaled[C.CONTINUOUS] = scaler.transform(X_test[C.CONTINUOUS])

# Scaler'ı kaydet (Streamlit'te yeni input gelince aynı transform uygulanacak)
joblib.dump(scaler, C.MODELS_DIR / "scaler.joblib")

print(f"Scaler models/scaler.joblib altına kaydedildi.")
print("\nScale edilmiş train (sürekli kolonlar) — ilk 5 satır:")
X_train_scaled[C.CONTINUOUS].head()

Scaler models/scaler.joblib altına kaydedildi.

Scale edilmiş train (sürekli kolonlar) — ilk 5 satır:


,age,trestbps,chol,thalach,oldpeak
0,-0.683973,-0.340150,0.642509,0.703161,-0.417862
1,0.090125,0.044957,0.372542,0.209545,-0.936193
2,-0.020461,0.173327,0.957470,0.389041,-0.936193
3,-0.020461,-1.367103,1.429911,0.254419,-0.936193
4,0.311296,0.686803,-0.099900,-1.226429,-0.728860


In [4]:
# Scale edilmiş veriyi de kaydedelim (sonraki notebook bunu yükleyecek)
X_train_scaled.to_csv(C.PROC_DIR / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(C.PROC_DIR / "X_test_scaled.csv", index=False)
print("Scaled veriler data/processed altına kaydedildi.")

Scaled veriler data/processed altına kaydedildi.


## 3. Yöntem 1 — SelectKBest (ANOVA F-test)

Her özelliğin hedef ile arasındaki ANOVA F-istatistiğini hesaplar; en yüksek
F-skoruna sahip K özelliği seçer. Hızlı, modelden bağımsız ve scale'e duyarsız.


In [5]:
K = C.TOP_K_FEATURES

selector_kbest = SelectKBest(score_func=f_classif, k=K)
selector_kbest.fit(X_train_scaled, y_train)

kbest_features = X_train_scaled.columns[selector_kbest.get_support()].tolist()
kbest_scores = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "f_score": selector_kbest.scores_,
}).sort_values("f_score", ascending=False)
print(f"SelectKBest (top-{K}):")
print(kbest_scores.head(K).to_string(index=False))

SelectKBest (top-8):
 feature   f_score
thal_7.0 82.636071
    cp_4 70.011685
 exang_1 48.439032
 thalach 46.000134
 oldpeak 42.196843
 slope_2 31.565222
   sex_1 27.013583
    cp_2 20.041533


## 4. Yöntem 2 — Recursive Feature Elimination (RFE)

LogReg eğitilir, en zayıf katsayılı özellik düşürülür, kalanla tekrar eğitilir... K özellik kalana kadar tekrarlanır.


In [6]:
base = LogisticRegression(max_iter=2000, random_state=C.RANDOM_STATE, class_weight="balanced")
rfe = RFE(estimator=base, n_features_to_select=K, step=1)
rfe.fit(X_train_scaled, y_train)

rfe_features = X_train_scaled.columns[rfe.support_].tolist()
rfe_rank = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "rfe_rank": rfe.ranking_,
}).sort_values("rfe_rank")
print(f"RFE (top-{K}, rank=1 olanlar):")
print(rfe_rank.head(K).to_string(index=False))

RFE (top-8, rank=1 olanlar):
 feature  rfe_rank
thal_7.0         1
  ca_1.0         1
 slope_2         1
 exang_1         1
  ca_2.0         1
    cp_4         1
   sex_1         1
  ca_3.0         1


## 5. Yöntem 3 — Embedded (Random Forest importance)

RF her ağaçta hangi özellikten ne kadar information gain elde edildiğini ortalayıp önem skoru üretir.


In [7]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=C.RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
rf.fit(X_train_scaled, y_train)

rf_importance = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "rf_importance": rf.feature_importances_,
}).sort_values("rf_importance", ascending=False)
rf_features = rf_importance.head(K)["feature"].tolist()

print(f"Random Forest (top-{K}):")
print(rf_importance.head(K).to_string(index=False))

Random Forest (top-8):
 feature  rf_importance
 thalach       0.127363
thal_7.0       0.123054
     age       0.099837
 oldpeak       0.097726
    cp_4       0.094638
    chol       0.091451
trestbps       0.076914
 exang_1       0.053120


## 6. Karşılaştırma tablosu (CSV + Markdown çıktısı)

In [8]:
all_features = sorted(set(X_train_scaled.columns))
comparison = pd.DataFrame({
    "feature": all_features,
    "SelectKBest": [int(f in kbest_features) for f in all_features],
    "RFE": [int(f in rfe_features) for f in all_features],
    "RandomForest": [int(f in rf_features) for f in all_features],
})
comparison["vote_count"] = comparison[["SelectKBest", "RFE", "RandomForest"]].sum(axis=1)
comparison = comparison.sort_values(["vote_count", "feature"], ascending=[False, True]).reset_index(drop=True)

U.save_table(comparison, C.METRICS_DIR / "feature_selection_comparison")

print("feature_selection_comparison.csv ve .md outputs/metrics altına kaydedildi.\n")
print(comparison.to_string(index=False))

feature_selection_comparison.csv ve .md outputs/metrics altına kaydedildi.

  feature  SelectKBest  RFE  RandomForest  vote_count
     cp_4            1    1             1           3
  exang_1            1    1             1           3
 thal_7.0            1    1             1           3
  oldpeak            1    0             1           2
    sex_1            1    1             0           2
  slope_2            1    1             0           2
  thalach            1    0             1           2
      age            0    0             1           1
   ca_1.0            0    1             0           1
   ca_2.0            0    1             0           1
   ca_3.0            0    1             0           1
     chol            0    0             1           1
     cp_2            1    0             0           1
 trestbps            0    0             1           1
     cp_3            0    0             0           0
    fbs_1            0    0             0           0
restec

## 7. Görsel — Venn diyagramı

In [9]:
try:
    from matplotlib_venn import venn3
    fig, ax = plt.subplots(figsize=(7, 7))
    venn3(
        [set(kbest_features), set(rfe_features), set(rf_features)],
        set_labels=("SelectKBest", "RFE", "RandomForest"),
        ax=ax,
    )
    ax.set_title(f"Özellik seçimi yöntemlerinin kesişimi (top-{K})")
    U.savefig(fig, C.FIG_DIR / "05_feature_selection_venn.png")
    print("Kaydedildi → outputs/figures/05_feature_selection_venn.png")
except ImportError:
    print("matplotlib-venn yüklü değil; pip install matplotlib-venn ile kurabilirsin.")

Kaydedildi → outputs/figures/05_feature_selection_venn.png


## 8. Görsel — Oy sayısına göre yatay bar plot

In [10]:
plot_df = comparison[comparison["vote_count"] > 0].sort_values("vote_count")
fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(plot_df))))
colors = ["#D9534F" if v == 3 else "#F0AD4E" if v == 2 else "#5BC0DE" for v in plot_df["vote_count"]]
ax.barh(plot_df["feature"], plot_df["vote_count"], color=colors, edgecolor="black")
ax.set_xlabel("Kaç yöntem tarafından seçildi (0–3)")
ax.set_xticks([0, 1, 2, 3])
ax.set_title("Özellik seçimi — yöntem oy sayısı")
U.savefig(fig, C.FIG_DIR / "06_feature_selection_votes.png")
print("Kaydedildi → outputs/figures/06_feature_selection_votes.png")

Kaydedildi → outputs/figures/06_feature_selection_votes.png


## 9. Ortak / oybirliği özelliklerini belirle

Strateji: **2 ve daha fazla yöntem tarafından seçilen** özelliklerle devam et.
3 yöntem ortaklığı çok daraltırsa modele yeterli sinyal kalmayabilir.

In [11]:
selected = comparison[comparison["vote_count"] >= 2]["feature"].tolist()
print(f"Seçilen özellik sayısı (>= 2 oy): {len(selected)}")
print(selected)

# Sonraki notebook'un kullanması için kaydet
joblib.dump(selected, C.MODELS_DIR / "selected_features.joblib")
print("\nmodels/selected_features.joblib altına kaydedildi.")

Seçilen özellik sayısı (>= 2 oy): 7
['cp_4', 'exang_1', 'thal_7.0', 'oldpeak', 'sex_1', 'slope_2', 'thalach']

models/selected_features.joblib altına kaydedildi.


## Çıktılar:

- `models/scaler.joblib`, `models/selected_features.joblib`
- `data/processed/X_train_scaled.csv`, `X_test_scaled.csv`
- `outputs/metrics/feature_selection_comparison.{csv,md}`
- `outputs/figures/05_feature_selection_venn.png`, `06_feature_selection_votes.png`

**Sonraki adım:** `03_modeling.ipynb`